# Model & DB loading

In [1]:
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoProcessor
from FlagEmbedding import BGEM3FlagModel

# load the model and processor
ckpt_siglip2 = "google/siglip2-giant-opt-patch16-384"
ckpt_bgem3 = "BAAI/bge-m3"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

processor_siglip2 = AutoProcessor.from_pretrained(ckpt_siglip2)
model_siglip2 = AutoModel.from_pretrained(
    ckpt_siglip2,
    device_map="auto",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2" # Install flash-attn before using this
).eval()
model_bgem3 = BGEM3FlagModel(
    ckpt_bgem3,
    normalize_embeddings=True,
    use_bf16=True,
    devices=device,
    return_dense=True,
    return_sparse=True,
    # return_colbert_vecs=True,
    model_kwargs={"attn_implementation": "flash_attention_2"}
)

C:\Users\Bui Thien Nghia\AppData\Roaming\Python\Python314\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [2]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url='http://localhost:6333')

# Benchmark utils definition

In [ ]:
def is_correct(candidate: list, ground_truth: list, tolerance=1):
    # [vid_id, frame_id]
    try:
        if candidate[0] != ground_truth[0]: return False
        return abs(candidate[1] - ground_truth[1]) <= tolerance
    except Exception: return False

def calculate_metrics(candidate_list, ground_truth, k_list=[1, 5, 10, 50], tolerance=1):
    metrics = {f"Result@{k}": 0 for k in k_list}
    metrics["MRR"] = 0

    ranking = 0
    for i, candidate in enumerate(candidate_list):
        verdict = is_correct(candidate, ground_truth)
        if not verdict: continue
        ranking = i + 1
        break

    if not ranking: return metrics

    metrics["MRR"] = 1 / ranking
    for k in k_list: metrics[f"Result@{k}"] = ranking <= k
    return metrics

# Query encoding

In [ ]:
QUERY = "Đây là hình ảnh về một buổi lễ hợp long cầu Mỹ Thuận 2 đang diễn ra long trọng ngay trên công trường của cây cầu dây văng. Phía sau là tấm băng rôn lớn mang dòng chữ "LỄ HỢP LONG CẦU MỸ THUẬN 2". Phía trước, các đại biểu và công nhân đội mũ bảo hộ đang thực hiện nghi thức xúc cát, xung quanh là những chùm pháo hoa sáng rực được bắn lên để chúc mừng."
inputs = processor_siglip2(text=QUERY, padding='max_length', truncation=True, return_tensors='pt')

with torch.no_grad():
    query_dense_siglip2 = model_siglip2.get_text_features(**inputs.to(device)).pooler_output
    query_dense_siglip2 = F.normalize(query_dense_siglip2, dim=-1)

query_dense_siglip2 = query_dense_siglip2.to('cpu').squeeze().tolist()

query_all_bgem3 = model_bgem3.encode(QUERY)
query_dense_bgem3 = query_all_bgem3['dense_vecs'].squeeze().tolist()
query_sparse_bgem3 = models.SparseVector(
    indices=np.array(list(query_all_bgem3['lexical_weights'].keys()), dtype=np.int32).tolist(),
    values=np.array(list(query_all_bgem3['lexical_weights'].values()), dtype=np.float32).tolist(),
)

In [ ]:
COLLECTION_NAME = 'test'
LIMIT = 50

prefetch = [
    models.Prefetch(
        query=query_dense_siglip2,
        using="dense_siglip2",
        limit=LIMIT * 2
    ),
    models.Prefetch(
        query=query_dense_bgem3,
        using="dense_bgem3",
        limit=LIMIT * 2
    ),
    models.Prefetch(
        query=query_sparse_bgem3,
        using='sparse_bgem3',
        limit=LIMIT * 2
    )
]

# Retrieval test

In [159]:
from IPython.display import display
import ipywidgets as widgets

hits = client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=prefetch,
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=LIMIT
).points

relevant_img_paths = [widgets.Image(value=open(hit.payload['img_path'], 'rb').read(), format='png', width=315) for hit in hits]
cols = 5
for i in range(0, len(relevant_img_paths), cols):
    if i + cols > len(relevant_img_paths):
        images = relevant_img_paths[i:len(relevant_img_paths)]
    else:
        images = relevant_img_paths[i:i+cols]

    row = widgets.HBox(images)
    display(row)
    print([hit.payload['img_path'].split('\\', 5)[-1].replace('\\', '_').replace(".jpg", '') for hit in hits[i:i+cols]])

['L01_V021_150', 'L01_V023_026', 'L01_V031_047', 'L01_V010_132', 'L01_V021_117']


['L01_V007_050', 'L01_V010_104', 'L01_V023_260', 'L01_V015_147', 'L01_V008_208']


['L01_V025_123', 'L01_V027_046', 'L01_V019_132', 'L01_V021_118', 'L01_V017_137']


['L01_V007_205', 'L01_V023_259', 'L01_V030_205', 'L01_V023_025', 'L01_V029_218']


['L01_V005_002', 'L01_V003_204', 'L01_V013_002', 'L01_V009_132', 'L01_V029_139']


['L01_V020_189', 'L01_V018_187', 'L01_V009_063', 'L01_V029_217', 'L01_V008_206']


['L01_V015_152', 'L01_V014_112', 'L01_V011_134', 'L01_V020_071', 'L01_V009_080']


['L01_V008_241', 'L01_V013_151', 'L01_V006_116', 'L01_V030_268', 'L01_V019_133']


['L01_V031_304', 'L01_V021_115', 'L01_V004_098', 'L01_V024_033', 'L01_V015_008']


['L01_V024_209', 'L01_V002_052', 'L01_V026_136', 'L01_V024_088', 'L01_V031_068']


In [161]:
QUERY = "Tiếp đến là góc chụp cận cảnh của một người đàn ông lớn tuổi có làn da ngăm đen, gương mặt đang cúi xuống tập trung làm việc. Ông mặc một chiếc áo thun màu xanh lam và đội chiếc nón tai bèo có dây quai màu đen cũng tone màu xanh lam."
ANS = "L01_V007_186"
COLLECTION_NAME = 'test'
LIMIT = 50

inputs = processor_siglip2(text=QUERY, padding='max_length', truncation=True, return_tensors='pt')

with torch.no_grad():
    query_dense_siglip2 = model_siglip2.get_text_features(**inputs.to(device)).pooler_output
    query_dense_siglip2 = F.normalize(query_dense_siglip2, dim=-1)

query_dense_siglip2 = query_dense_siglip2.to('cpu').squeeze().tolist()

query_all_bgem3 = model_bgem3.encode(QUERY)
query_dense_bgem3 = query_all_bgem3['dense_vecs'].squeeze().tolist()
query_sparse_bgem3 = models.SparseVector(
    indices=np.array(list(query_all_bgem3['lexical_weights'].keys()), dtype=np.int32).tolist(),
    values=np.array(list(query_all_bgem3['lexical_weights'].values()), dtype=np.float32).tolist(),
)

prefetch = [
    models.Prefetch(
        query=query_dense_siglip2,
        using="dense_siglip2",
        limit=LIMIT * 2
    ),
    models.Prefetch(
        query=query_dense_bgem3,
        using="dense_bgem3",
        limit=LIMIT * 2
    ),
    models.Prefetch(
        query=query_sparse_bgem3,
        using='sparse_bgem3',
        limit=LIMIT * 2
    )
]

prefetch_combs = [
    [prefetch[0]], # D1
    [prefetch[0], prefetch[1]], # D1 + D2
    [prefetch[0], prefetch[2]], # D1 + S2
    prefetch # D1 + D2 + S2
]

for i, p in enumerate(prefetch_combs):
    hits = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=p,
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=LIMIT
    ).points

    candidate_list = [hit.payload['img_path'].split('\\', 5)[-1].replace('\\', '_').replace(".jpg", '') for hit in hits]
    rank = 999999
    # for j in range(-1, 2):
    #     try:
    #         ans_split = ANS.split('_')
    #         ans_frame_id = int(ans_split[-1])
    #         ans_lenient = f'{ans_split[0]}_{ans_split[1]}_{(ans_frame_id + j):03d}'
    #         rank = min(rank, candidate_list.index(ans_lenient))
    #     except ValueError: continue
    try:
        rank = candidate_list.index(ANS)
    except ValueError: pass
    
    if rank == 999999:
        print(f"Combination {i + 1} failed the test")
    else:
        print(f"Combination {i + 1} yielded rank {rank + 1}")

Combination 1 yielded rank 35
Combination 2 failed the test
Combination 3 failed the test
Combination 4 failed the test


1 2 1 3
8 13 16 19
3 6 6 8
4 7 8 11

10 20 18 26
9 17 16 25
100 100 34 30
100 100 100 100